# 05 — Temporal Backtesting: Blocked Cross-Validation

Evaluates recommendation quality using blocked (non-shuffled) temporal splits — no data leakage.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')
%matplotlib inline

## Build Synthetic Zone-Year Panel


In [ ]:
from src.models.cmf_score import ScoreComponents, compute_opening_score, score_zone_for_concept
from src.validation.backtesting import build_blocked_splits, apply_temporal_split, evaluate_top_k

rng = np.random.default_rng(42)
zones = [f'Z{i:02d}' for i in range(1, 21)]
years = list(range(2018, 2025))

rows = []
for z in zones:
    base_score = rng.random() * 0.5 + 0.2  # Each zone has a latent quality
    for yr in years:
        noise = rng.normal(0, 0.05)
        rows.append({'zone_id': z, 'year': yr,
                     'opportunity_score': float(np.clip(base_score + noise + (yr - 2018) * 0.01, 0, 1))})

panel = pd.DataFrame(rows)
print(f'Panel: {len(panel)} rows, {panel["year"].nunique()} years, {panel["zone_id"].nunique()} zones')
print(panel.head(6))

## Build Blocked Temporal Splits


In [ ]:
splits = build_blocked_splits(panel, time_col='year', min_train_periods=3, test_size=1)
print(f'Number of splits: {len(splits)}')
for i, s in enumerate(splits):
    print(f'  Split {i+1}: train={list(s.train_periods)}, test={list(s.test_periods)}')

## Recall@5 Across Splits


In [ ]:
recall_scores = []

for split in splits:
    train_df, test_df = apply_temporal_split(panel, 'year', split)

    # Train: rank zones by mean score in training window
    train_ranked = (
        train_df.groupby('zone_id')['opportunity_score']
        .mean()
        .sort_values(ascending=False)
        .index.tolist()
    )
    recommended_top5 = train_ranked[:5]

    # Test: actual top-5 zones in test period
    test_ranked = (
        test_df.groupby('zone_id')['opportunity_score']
        .mean()
        .sort_values(ascending=False)
        .index.tolist()
    )
    actual_top5 = test_ranked[:5]

    score = evaluate_top_k(recommended_top5, actual_top5, k=5)
    recall_scores.append({'split': f'{split.train_periods[-1]}→{split.test_periods[0]}', 'recall_at_5': score})

results_df = pd.DataFrame(recall_scores)
print(results_df)
print(f'\nMean Recall@5: {results_df["recall_at_5"].mean():.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(results_df['split'], results_df['recall_at_5'], marker='o', color='#4C72B0')
ax.axhline(results_df['recall_at_5'].mean(), color='#DD8452', linestyle='--', label=f'Mean {results_df["recall_at_5"].mean():.2f}')
ax.set_ylim(0, 1.05)
ax.set_xlabel('Split (train end → test year)')
ax.set_ylabel('Recall@5')
ax.set_title('Temporal Backtest — Recall@5 Across Splits')
ax.legend()
plt.tight_layout()
plt.show()

## Explicit Cutoff Split (2022 / 2023)


In [ ]:
from src.validation.backtesting import train_test_split_by_cutoff

train_df, test_df = train_test_split_by_cutoff(panel, 'year', train_end=2022, test_start=2023)
print(f'Train years: {sorted(train_df["year"].unique())}')
print(f'Test years:  {sorted(test_df["year"].unique())}')

## Feature Ablation Study


In [ ]:
from src.models.cmf_score import ScoreComponents, compute_opening_score

rng3 = np.random.default_rng(99)
abla_rows = []
for zone_id in zones:
    feats = {
        'healthy_gap_score': float(rng3.uniform(0.2, 0.9)),
        'subtype_gap_score': float(rng3.uniform(0.2, 0.9)),
        'merchant_viability_score': float(rng3.uniform(0.3, 0.8)),
        'competition_penalty': float(rng3.uniform(0.1, 0.6)),
    }
    full_score = compute_opening_score(ScoreComponents(**feats))
    # Ablate each feature — set it to 0
    for ablated_feat in feats:
        ablated = {k: (0.0 if k == ablated_feat else v) for k, v in feats.items()}
        ablated_score = compute_opening_score(ScoreComponents(**ablated))
        abla_rows.append({'zone_id': zone_id, 'ablated': ablated_feat,
                           'delta': full_score - ablated_score})

ablate_df = pd.DataFrame(abla_rows)
mean_impact = ablate_df.groupby('ablated')['delta'].mean().sort_values(ascending=False)
print('Mean score drop when ablating each feature:')
print(mean_impact.round(4))

fig, ax = plt.subplots(figsize=(6, 3))
mean_impact.plot.barh(ax=ax, color='#C44E52', edgecolor='white')
ax.set_xlabel('Mean score drop')
ax.set_title('Feature Ablation — Impact on CMF Score')
plt.tight_layout()
plt.show()

## Summary

- Blocked temporal splits prevent data leakage — essential for time-ordered evaluation.
- Mean Recall@5 measures whether zones ranked high in training windows stay high in the next period.
- `merchant_viability_score` and `subtype_gap_score` drive the most score variance in ablations.
- Next step: add concordance index from the survival model into the evaluation loop.
